# Demonstration of how to pipelines between X22 model and RADMC-3D

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append('..')
from radmc.setup import radmc3d_setup
from radmc.simulate import generate_simulation
from radmc.plot import generate_plot
import os 

## Generate disk model and RADMC3D-required files

In [5]:
amax        = 0.1 # maximum grain size in mm
mstar       = 0.1 # stellar mass in solar masses
mdot        = 1e-7 # accretion rate in solar masses per year
rd          = 60 # disk radius in AU
Toomre_Q    = 1.5 # Toomre Q parameter
l_star      = 1.0 # stellar luminosity in solar luminosities
heating     = 'radiation' # heating mechanism

In [ ]:
model = radmc3d_setup(silent=False)
model.get_mastercontrol(filename=None,
                        comment=None,
                        incl_dust=1,
                        incl_lines=1,
                        nphot=1000000,
                        nphot_scat=10000000,
                        scattering_mode_max=2,
                        istar_sphere=1,
                        num_cpu=None,
                        modified_random_walk = 1
                        )
model.get_linecontrol(filename=None,
                    methanol='ch3oh leiden 0 0 0')
model.get_continuumlambda(filename=None,
                        comment=None,
                        lambda_micron=None,
                        append=False)

model.get_diskcontrol(  d_to_g_ratio    = 0.01,
                        a_max           = amax, # mm
                        Mass_of_star    = mstar, # Msun
                        Accretion_rate  = mdot, # Msun/yr
                        Radius_of_disk  = rd,   # AU
                        Q               = Toomre_Q, # Toomre Q
                        NR    =200,
                        NTheta=200,
                        NPhi  =20,
                        )
model.get_vfieldcontrol(Kep=True,
                        vinfall=0.5, # the infall velocity (unit: Keperian velocity)
                        Rcb=None, # the centrifugal barrier
                        outflow=None)
model.get_heatcontrol(L_star=l_star, # Lsun
                      R_star=1,
                      heat=heating) # radiation/accretion
model.get_gasdensitycontrol(abundance=1e-10, # abundance of CH3OH compared to H2
                            snowline=100, # snowline temperature
                            enhancement=1e5, # enhancement factor of abundance inside snowline
                            gas_inside_rcb=True)

## RADMC3D simulations

In [7]:
simulation = generate_simulation(save_out=True, save_npz=True)

simulate_mutual_parms = {
    "incl"      : 73,
    "line"      : 240,
    "npix"      : 500,
    "sizeau"    : 200,
    "v_width"   : 10,
    "vkms"      : 0,
    "v_width"   : 10,
    "dir"       : './test/',
    "fname"     : 'test',
}


In [ ]:
simulation.generate_cube(
    nodust=False, scat=True, extract_gas=True,
    nlam=11,
    **simulate_mutual_parms
)

In [ ]:
simulation.generate_cube(
    nodust=False, scat=True, extract_gas=True,
    nlam=50,
    **simulate_mutual_parms
)

In [ ]:
simulation.generate_continuum(
   scat=True,
   wav=1300,
   **simulate_mutual_parms
)

In [ ]:
simulation.generate_sed(
    scat=True,
    freq_min=5e1, freq_max=5e2, nlam=10,
    **simulate_mutual_parms
)


In [ ]:
simulation.generate_line_spectrum(
    nodust=False, scat=True, extract_gas=True,
    nlam=10,
    **simulate_mutual_parms
)